In [1]:
import kagglehub
path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")


/run/media/islam/6cd0d34e-fc8d-4fdf-952e-4c1c86fd93e8/FCAI/Level3/2/supervised/Autonomous-Vehicle-Perception-Module/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from tqdm import tqdm
import numpy as np
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [4]:
TRAIN_DIR   = os.path.join(path, "Train")
TEST_DIR    = os.path.join(path, "Test")
TEST_CSV    = os.path.join(path, "Test.csv")

In [5]:
NUM_CLASSES = 43
CLASS_NAMES = {
    0:'Speed limit (20)',    1:'Speed limit (30)',   2:'Speed limit (50)',
    3:'Speed limit (60)',    4:'Speed limit (70)',   5:'Speed limit (80)',
    6:'End speed limit(80)', 7:'Speed limit (100)',  8:'Speed limit (120)',
    9:'No passing',         10:'No passing >3.5t',  11:'Right-of-way',
    12:'Priority road',     13:'Yield',              14:'Stop',
    15:'No vehicles',       16:'No >3.5t vehicles', 17:'No entry',
    18:'General caution',   19:'Danger curve left', 20:'Danger curve right',
    21:'Double curve',      22:'Bumpy road',         23:'Slippery road',
    24:'Road narrows',      25:'Road work',          26:'Traffic signals',
    27:'Pedestrians',       28:'Children crossing',  29:'Bicycles crossing',
    30:'Ice/snow',          31:'Wild animals',       32:'End speed+passing',
    33:'Turn right ahead',  34:'Turn left ahead',    35:'Ahead only',
    36:'Go straight/right', 37:'Go straight/left',   38:'Keep right',
    39:'Keep left',         40:'Roundabout',          41:'End no passing',
    42:'End no passing >3.5t'
}

In [6]:
print("Loading data from folder structure...")

train_paths = []
train_labels = []

# Loop through each class folder (0 to 42)
for class_id in tqdm(range(NUM_CLASSES), desc="Scanning class folders"):
    class_folder = os.path.join(TRAIN_DIR, str(class_id))
    if os.path.exists(class_folder):
        # Get all PNG files in the folder
        for img_file in os.listdir(class_folder):
            if img_file.endswith('.png'):
                img_path = os.path.join(class_folder, img_file)
                train_paths.append(img_path)
                train_labels.append(class_id)

print(f"\nTotal training images found: {len(train_paths):,}")

Loading data from folder structure...


Scanning class folders: 100%|██████████| 43/43 [00:00<00:00, 180.14it/s]


Total training images found: 39,209


In [7]:
#print the shape of the first image to verify loading
if train_paths:
    sample_image = Image.open(train_paths[50])
    print(f"Sample image size: {sample_image.size} (width x height)")

Sample image size: (43, 43) (width x height)


In [8]:
#load images into train data frame
data_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

In [9]:
class MyImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image lazily (only when needed)
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [10]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Split paths and labels (80% train, 20% validation)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.2, random_state=42
)

# Create Datasets
train_ds = MyImageDataset(train_paths, train_labels, transform=data_transforms)
val_ds = MyImageDataset(val_paths, val_labels, transform=data_transforms)

# Create Loaders
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)


In [11]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        # Feature Extraction
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        # Classification Head
        # If input 64x64, after two 2x2 pools it becomes 32x32
        self.fc1 = nn.Linear(32 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) # Feature extraction
        x = self.pool(F.relu(self.conv2(x))) # Feature extraction
        x = x.view(x.size(0), -1)            # Flatten for the NN
        x = F.relu(self.fc1(x))              # Classification
        x = self.fc2(x)
        return x

In [13]:
#use SimpleCNN to train the model
import torch.optim as optim
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
def train_CNN(num_epochs, train_loader, val_loader, model, criterion, optimizer):

    for epoch in range(num_epochs):
        # --- TRAINING PHASE ---
        model.train() # Set to training mode
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass: Get predictions
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass: Optimize the weights
            optimizer.zero_grad() # Clear old gradients
            loss.backward()      # Calculate new gradients
            optimizer.step()     # Update weights

            train_loss += loss.item()

        # --- VALIDATION PHASE ---
        model.eval() # Set to evaluation mode (turns off dropout, etc.)
        val_correct = 0
        val_total = 0

        with torch.no_grad(): # Disable gradient calculation (saves memory/time)
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)

                # The output is a raw score (logit); pick the highest score
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss/len(train_loader):.4f} | Val Acc: {100 * val_correct / val_total:.2f}%")

In [15]:
train_CNN(num_epochs=10, train_loader=train_loader, val_loader=val_loader, model=model, criterion=criterion, optimizer=optimizer)

Epoch [1/10] | Train Loss: 0.6714 | Val Acc: 94.90%
Epoch [2/10] | Train Loss: 0.1030 | Val Acc: 97.55%
Epoch [3/10] | Train Loss: 0.0597 | Val Acc: 97.77%
Epoch [4/10] | Train Loss: 0.0411 | Val Acc: 97.58%
Epoch [5/10] | Train Loss: 0.0395 | Val Acc: 97.91%
Epoch [6/10] | Train Loss: 0.0335 | Val Acc: 98.19%
Epoch [7/10] | Train Loss: 0.0196 | Val Acc: 97.82%
Epoch [8/10] | Train Loss: 0.0214 | Val Acc: 97.50%
Epoch [9/10] | Train Loss: 0.0202 | Val Acc: 97.83%
Epoch [10/10] | Train Loss: 0.0182 | Val Acc: 97.83%
